我们专门做一个**能直观看出 Expert 自动分工**的实验。

任务设为：

$$
y=
\begin{cases}
2x+1,&x<0\\
-3x+1,&x\geq0
\end{cases}
$$

设置 **2 个线性 Expert + 1 个 Gate**。训练过程中**不给 Expert 任何分工标签**，只优化 mixture NLL。训练完成后，在推理阶段分别打印两个 Expert 的输出、Router 概率和最终选择的 Expert，从而验证是否自动形成专业化。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)


# ============================================================
# 1. Expert
# 每个 Expert 都是一个简单的线性模型：
# mu_i = w_i * x + b_i
# ============================================================
class Expert(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        return self.fc(x)


# ============================================================
# 2. Gate / Router
# 根据输入 x 输出两个 Expert 的选择概率
#
# logits -> softmax -> [p1, p2]
# ============================================================
class Gate(nn.Module):
    def __init__(self, num_experts=2):
        super().__init__()
        self.fc = nn.Linear(1, num_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=-1)
        return logits, probs


# ============================================================
# 3. Mixture of Experts
# ============================================================
class MoE(nn.Module):
    def __init__(self, num_experts=2):
        super().__init__()

        self.num_experts = num_experts

        self.gate = Gate(num_experts)

        self.experts = nn.ModuleList([
            Expert() for _ in range(num_experts)
        ])

    def forward(self, x):

        # ---------- Router ----------
        logits, probs = self.gate(x)       # [B, K]

        # ---------- Experts ----------
        outputs = []

        for expert in self.experts:
            outputs.append(expert(x))      # 每个都是 [B,1]

        # [B,1] * K -> [B,K]
        mus = torch.cat(outputs, dim=1)

        return logits, probs, mus


## logsumexp
---

## 1. 核心公式

$$
\text{logsumexp}(x_1,\dots,x_K)
=
\log \sum_{i=1}^K e^{x_i}
$$

它的作用就是：  
**在 log 域里做“求和”**。

普通求和是：

$$
S = \sum_i a_i
$$

如果已知每个 $a_i$ 的对数 $x_i = \log a_i$，那么：

$$
\log S = \log \sum_i a_i = \log \sum_i e^{x_i}
$$

这就是 logsumexp。

---

## 2. 为什么需要 logsumexp？

直接算 $\log(\sum_i e^{x_i})$ 会有数值问题：

- 如果某个 $x_i$ 很大，比如 $x_i = 1000$，则 $e^{1000}$ 会溢出成 `inf`。
- 如果所有 $x_i$ 都很小，比如 $-1000$，则 $e^{-1000}$ 会下溢成 `0`，丢失精度。

logsumexp 的标准稳定实现是：

$$
\text{logsumexp}(x)
=
m + \log \sum_i e^{x_i - m}
$$

其中：

$$
m = \max_i x_i
$$

因为减去最大值后，$x_i - m \le 0$，所以 $e^{x_i - m} \in (0, 1]$，不会溢出。  
而 $m$ 最后加回来，结果不变。

---

## 3. PyTorch API：`torch.logsumexp`

### 函数签名

```python
torch.logsumexp(input, dim, keepdim=False)
```

- `input`：输入张量
- `dim`：在哪个维度上做 logsumexp，可以是 `int` 或 `tuple`
- `keepdim`：是否保持被归约的维度，默认为 `False`

返回值：对指定维度做 logsumexp 后的张量。

### 一维例子

```python
import torch

x = torch.tensor([1.0, 2.0, 3.0])
result = torch.logsumexp(x, dim=0)

# 等价于：
# torch.log(torch.sum(torch.exp(x)))
print(result)  # tensor(3.4076)
```

验证：

$$
\log(e^1 + e^2 + e^3) = \log(2.718 + 7.389 + 20.085) = \log(30.192) \approx 3.4076
$$

### 二维例子

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [0.0, 1.0, 2.0]
])

result = torch.logsumexp(x, dim=1)
print(result)  # tensor([3.4076, 2.4076])
```

这里 `dim=1` 表示对每一行做 logsumexp：

- 第一行：$\log(e^1 + e^2 + e^3)$
- 第二行：$\log(e^0 + e^1 + e^2)$

### 与普通写法的对比

```python
# 不稳定写法
naive = torch.log(torch.sum(torch.exp(x), dim=1))

# 稳定写法
stable = torch.logsumexp(x, dim=1)

# 结果相同，但 stable 更安全
```

---

## 4. 在 MoE loss 中的角色

回到之前的 MoE loss：

```python
log_mixture = torch.logsumexp(
    log_p + log_q,
    dim=1
)
```

这里：

- `log_p` = $\log p_i$
- `log_q` = $\log q_i$
- `log_p + log_q` = $\log(p_i q_i)$

于是：

$$
\text{logsumexp}(\log p_i + \log q_i)
=
\log \sum_i e^{\log p_i + \log q_i}
=
\log \sum_i p_i q_i
$$

这正是混合模型的对数密度：

$$
\log P(y_b) = \log \sum_i p_{b,i} q_{b,i}
$$

所以 `torch.logsumexp` 帮我们在 log 域里完成了“先乘后加”的混合计算，而且不会数值下溢。

---

## 5. 梯度直觉

对 `logsumexp` 求导：

$$
\frac{\partial}{\partial x_j} \text{logsumexp}(x)
=
\frac{e^{x_j}}{\sum_i e^{x_i}}
=
\text{softmax}(x)_j
$$

也就是说，logsumexp 的梯度就是 softmax。  
在 MoE 中，这正好对应每个专家的后验责任：

$$
r_i = \frac{p_i q_i}{\sum_j p_j q_j}
$$

所以梯度会自然地按责任大小分配给各个专家。


In [ ]:

# ============================================================
# 4. 原始概率型 MoE Loss
#
# L = -log sum_i p_i * N(y; mu_i, sigma^2)
# ============================================================
def moe_loss(probs, mus, y, sigma=0.5):

    # [B,1] -> [B,K]
    y_expand = y.expand_as(mus)

    # --------------------------------------------------------
    # log q_i   (B,K)
    #
    # q_i = N(y; mu_i, sigma^2)
    #
    # log q_i =
    # - (y-mu_i)^2 / (2 sigma^2)
    # - log(sigma)
    # - 1/2 log(2pi)
    # --------------------------------------------------------
    log_q = (
        -0.5 * ((y_expand - mus) / sigma) ** 2
        - math.log(sigma)
        - 0.5 * math.log(2 * math.pi)
    )

    # log p_i  (B,K)
    log_p = torch.log(probs + 1e-8)

    # --------------------------------------------------------
    # log sum_i p_i q_i
    #
    # 使用 logsumexp 保证数值稳定,在维度1上进行求和
    # --------------------------------------------------------
    log_mixture = torch.logsumexp(
        log_p + log_q,
        dim=1
    )

    # Negative Log Likelihood
    loss = -log_mixture.mean()

    return loss


In [2]:

# ============================================================
# 5. 构造训练数据
#
#             2x + 1, x < 0
# y =
#            -3x + 1, x >= 0
#
# 注意：
# 我们没有告诉模型：
#   x < 0 -> Expert 1
#   x >=0 -> Expert 2
#
# 这个分工必须由 MoE 自己学出来
# ============================================================

N = 1000

x = torch.linspace(-5, 5, N).reshape(-1, 1)

y = torch.where(
    x < 0,
    2 * x + 1,
    -3 * x + 1
)


# ============================================================
# 6. 创建模型
# ============================================================

model = MoE(num_experts=2)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


# ============================================================
# 7. 训练
# ============================================================

epochs = 3000

for epoch in range(epochs):

    logits, probs, mus = model(x)

    loss = moe_loss(
        probs,
        mus,
        y,
        sigma=0.5
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 500 == 0:
        print(
            f"epoch = {epoch:4d}, "
            f"loss = {loss.item():.6f}"
        )


epoch =    0, loss = 81.453491
epoch =  500, loss = 1.608766
epoch = 1000, loss = 0.776782
epoch = 1500, loss = 0.337591
epoch = 2000, loss = 0.250920
epoch = 2500, loss = 0.240568


In [ ]:

# ============================================================
# 8. 查看两个 Expert 最终学到的函数
# ============================================================

print("\n================ Expert Parameters ================\n")

for i, expert in enumerate(model.experts):

    w = expert.fc.weight.item()
    b = expert.fc.bias.item()

    print(
        f"Expert {i}: "
        f"y = {w:.4f} * x + {b:.4f}"
    )



================ Expert Parameters ================

Expert 0: y = 1.9953 * x + 0.9846
Expert 1: y = -2.9945 * x + 0.9821


In [ ]:
# ============================================================
# 9. 推理测试
#
# 这里故意选择左右两侧的点：
#
# x < 0
# x > 0
#
# 看 Router 是否选择不同 Expert
# ============================================================

test_x = torch.tensor([
    [-4.0],
    [-2.0],
    [-1.0],
    [ 1.0],
    [ 2.0],
    [ 4.0]
])

# GT
test_y = torch.where(
    test_x < 0,
    2 * test_x + 1,
    -3 * test_x + 1
)


with torch.no_grad():

    logits, probs, mus = model(test_x)

    # Router选择概率最大的Expert
    selected_expert = torch.argmax(
        probs,
        dim=1
    )

    # 根据Router选择最终预测
    final_prediction = mus.gather(
        1,
        selected_expert.unsqueeze(1)
    )

# ============================================================
# 10. 打印推理结果
# ============================================================

print("\n================ Inference ================\n")

for i in range(len(test_x)):

    print(f"x = {test_x[i].item():5.1f}")

    print(
        f"GT       = {test_y[i].item():8.3f}"
    )

    print(
        f"Expert 0 = {mus[i,0].item():8.3f}"
    )

    print(
        f"Expert 1 = {mus[i,1].item():8.3f}"
    )

    print(
        f"Router   = "
        f"[{probs[i,0].item():.4f}, "
        f"{probs[i,1].item():.4f}]"
    )

    print(
        f"Selected = Expert "
        f"{selected_expert[i].item()}"
    )

    print(
        f"Prediction = "
        f"{final_prediction[i].item():8.3f}"
    )

    print("-" * 45)


================ Inference ================

x =  -4.0
GT       =   -7.000
Expert 0 =   -6.997
Expert 1 =   12.960
Router   = [1.0000, 0.0000]
Selected = Expert 0
Prediction =   -6.997
---------------------------------------------
x =  -2.0
GT       =   -3.000
Expert 0 =   -3.006
Expert 1 =    6.971
Router   = [1.0000, 0.0000]
Selected = Expert 0
Prediction =   -3.006
---------------------------------------------
x =  -1.0
GT       =   -1.000
Expert 0 =   -1.011
Expert 1 =    3.977
Router   = [0.9989, 0.0011]
Selected = Expert 0
Prediction =   -1.011
---------------------------------------------
x =   1.0
GT       =   -2.000
Expert 0 =    2.980
Expert 1 =   -2.012
Router   = [0.0011, 0.9989]
Selected = Expert 1
Prediction =   -2.012
---------------------------------------------
x =   2.0
GT       =   -5.000
Expert 0 =    4.975
Expert 1 =   -5.007
Router   = [0.0000, 1.0000]
Selected = Expert 1
Prediction =   -5.007
---------------------------------------------
x =   4.0
GT       =  -1


## 我们期待看到什么？

首先看 Expert 参数。

理想情况下，训练结束以后可能得到：

```text
================ Expert Parameters ================

Expert 0: y =  2.0012 * x + 1.0031
Expert 1: y = -2.9987 * x + 0.9975
```

或者反过来：

```text
Expert 0: y = -3x + 1
Expert 1: y =  2x + 1
```

**Expert 编号没有任何语义，所以两种结果完全等价。**

我们没有告诉：

```text
Expert 0 学 2x+1
Expert 1 学 -3x+1
```

但它们通过竞争自己产生了：

$$
\boxed{
Expert_A\rightarrow 2x+1
}
$$

$$
\boxed{
Expert_B\rightarrow -3x+1
}
$$

这就是我们要观察的 specialization。

---

## 推理阶段更加关键

假设最终：

$$
E_0(x)\approx2x+1
$$

$$
E_1(x)\approx-3x+1
$$

那么对于：

$$
x=-2
$$

GT：

$$
y=2(-2)+1=-3
$$

可能得到：

```text
x = -2.0

GT       = -3.000

Expert 0 = -3.002
Expert 1 =  7.010

Router   = [0.9981, 0.0019]

Selected = Expert 0

Prediction = -3.002
```

Router 明显选择：

$$
Expert_0
$$

而对于：

$$
x=2
$$

GT：

$$
y=-3(2)+1=-5
$$

可能得到：

```text
x = 2.0

GT       = -5.000

Expert 0 =  5.003
Expert 1 = -4.998

Router   = [0.0023, 0.9977]

Selected = Expert 1

Prediction = -4.998
```

Router 切换到了：

$$
Expert_1
$$

因此我们最终希望验证三件事情：

$$
\boxed{
\begin{array}{c}
Expert_0\text{ 学会其中一个线性函数}\\
Expert_1\text{ 学会另一个线性函数}\\
Router\text{ 学会在 }x=0\text{ 附近切换Expert}
\end{array}
}
$$

---

## 这段代码与前面公式是一一对应的

代码中的：

```python
logits, probs = self.gate(x)
```

对应：

$$
a_i\rightarrow
p_i=Softmax(a)_i
$$

Expert：

```python
mus = torch.cat(outputs, dim=1)
```

对应：

$$
\mu_i=E_i(x)
$$

Gaussian：

```python
log_q = -0.5 * ((y_expand - mus) / sigma) ** 2 + ...
```

对应：

$$
q_i
=
\mathcal N(y;\mu_i,\sigma^2)
$$

然后：

```python
log_mixture = torch.logsumexp(
    log_p + log_q,
    dim=1
)
```

对应：

$$
\log
\sum_i
p_iq_i
$$

最终：

```python
loss = -log_mixture.mean()
```

就是：

$$
\boxed{
L
=
-\frac1B
\sum_{n=1}^{B}
\log
\left[
\sum_i
p_i(x_n)
\mathcal N
(y_n;\mu_i(x_n),\sigma^2)
\right]
}
$$

最后一句：

```python
loss.backward()
```

就会自动产生我们上一轮详细推导的：

$$
\boxed{
\frac{\partial L}{\partial a_i}
=
p_i-r_i
}
$$

以及：

$$
\boxed{
\frac{\partial L}{\partial\mu_i}
=
r_i\frac{\mu_i-y}{\sigma^2}
}
$$

所以这个小实验的重点不是拟合一个分段函数本身，而是**亲眼看到：我们没有给 Expert 分工标签，但 mixture likelihood + responsibility 机制会让 Expert 自己形成分工，同时让 Router 学会什么输入应该交给谁。**